### Import Libraries

In [49]:
from ibm_watsonx_ai import APIClient
from llama_index.llms.ibm import WatsonxLLM
from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import AutoTokenizer
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex, DocumentSummaryIndex, KeywordTableIndex
from llama_index.core.retrievers import VectorIndexRetriever

In [14]:
def create_llm():
    
    try:
        
        tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
        
        llm = HuggingFaceLLM(
            model_name="meta-llama/Llama-3.2-1B-Instruct",
            tokenizer=tokenizer,
            max_new_tokens=256,
            context_window=2000,
            model_kwargs={
    
                "torch_dtype": "auto"
            }

        )
        
        print("LLM initialized using official LlamaIndex integration")
        return llm
        
    except Exception as e:
        print(f"Error initializing: {e}")
        print("Falling back to mock LLM for demonstration")
        


In [ ]:
llm = create_llm()

LLM initialized using official LlamaIndex integration


In [19]:
response = llm.complete("Hi")
print(response.text)

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


 there! I'm new to this community and I'm excited to share my latest creation, a unique and creative writing project that I've been working on.

I'd love to get your feedback and suggestions on my project. Please take a look and let me know what you think!

**Project Title:** "Echoes in the Attic"

**Genre:** Magical Realism/Short Story

**Synopsis:** "Echoes in the Attic" is a story about a young woman who inherits an old mansion from a distant relative she's never met. As she explores the dusty, cobweb-filled rooms, she discovers a mysterious book that seems to hold the key to unlocking the secrets of her family's past. But as she delves deeper into the book's secrets, she realizes that she's not alone in the attic... and that the echoes of the past are beginning to manifest in the present.

**Writing Style:** I've aimed for a lyrical, atmospheric writing style that blends the magical and the mundane. I've used vivid descriptions of old-fashioned objects, decaying furniture, and the 

In [23]:
embedding_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-m3"
)

In [37]:
Settings.llm = llm
Settings.embed_model=embedding_model

---

## Background

Before diving into the advanced retrieval techniques, let's understand the foundational concepts that make these retrievers powerful.

### What are Advanced Retrievers?

Advanced retrievers in LlamaIndex are sophisticated components that go beyond simple vector similarity search to provide more nuanced, context-aware, and intelligent information retrieval. They combine multiple techniques such as:

- **Semantic Understanding**: Using embeddings to understand meaning and context
- **Keyword Matching**: Precise term-based search for exact specifications
- **Hierarchical Context**: Maintaining relationships between different levels of information
- **Multi-Query Processing**: Generating and combining results from multiple query variations
- **Fusion Techniques**: Intelligently combining results from different retrieval methods

### Why are Advanced Retrievers Important?

1. **Improved Accuracy**: Advanced retrievers can find more relevant information by using multiple search strategies
2. **Better Context Preservation**: They maintain important relationships between pieces of information
3. **Reduced Hallucination**: More precise retrieval leads to more accurate AI responses
4. **Scalability**: Efficient retrieval strategies work better with large document collections
5. **Flexibility**: Different retrieval methods can be combined for optimal results

### Index Types Overview

Before exploring advanced retrievers, it's helpful to first understand the three main index types supported by LlamaIndex. Each is designed to support different retrieval scenarios:

**VectorStoreIndex:**
- Stores vector embeddings for each document chunk
- Best suited for semantic retrieval based on meaning
- Commonly used in LLM pipelines and RAG applications

**DocumentSummaryIndex:**
- Generates and stores summaries of documents at indexing time
- Uses summaries to filter documents before retrieving full content
- Especially useful for large and diverse document sets that cannot fit in the context window of an LLM

**KeywordTableIndex:**
- Extracts keywords from documents and maps them to specific content chunks
- Enables exact keyword matching for rule-based or hybrid search scenarios
- Ideal for applications requiring precise term matching

## Sample Data Setup

We'll use a collection of AI and machine learning documents to demonstrate different retrieval strategies.


In [27]:
# Sample data for the lab - AI/ML focused documents
SAMPLE_DOCUMENTS = [
    "Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.",
    "Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.",
    "Natural language processing enables computers to understand, interpret, and generate human language.",
    "Computer vision allows machines to interpret and understand visual information from the world.",
    "Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.",
    "Supervised learning uses labeled training data to learn a mapping from inputs to outputs.",
    "Unsupervised learning finds hidden patterns in data without labeled examples.",
    "Transfer learning leverages knowledge from pre-trained models to improve performance on new tasks.",
    "Generative AI can create new content including text, images, code, and more.",
    "Large language models are trained on vast amounts of text data to understand and generate human-like text."
]

# Consistent query examples used throughout the lab
DEMO_QUERIES = {
    "basic": "What is machine learning?",
    "technical": "neural networks deep learning", 
    "learning_types": "different types of learning",
    "advanced": "How do neural networks work in deep learning?",
    "applications": "What are the applications of AI?",
    "comprehensive": "What are the main approaches to machine learning?",
    "specific": "supervised learning techniques"
}

print(f"📄 Loaded {len(SAMPLE_DOCUMENTS)} sample documents")
print(f"🔍 Prepared {len(DEMO_QUERIES)} consistent demo queries")
for i, doc in enumerate(SAMPLE_DOCUMENTS[:3], 1):
    print(f"{i}. {doc}")
print("...")

📄 Loaded 10 sample documents
🔍 Prepared 7 consistent demo queries
1. Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
2. Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.
3. Natural language processing enables computers to understand, interpret, and generate human language.
...


Lets create 3 main indexes as a class:


In [55]:
class AdvanceRetrieverIndex:
    
    def __init__(self):
        print("Initializing Advanced Retrievers Lab...")
        self.documents = [Document(text=text) for text in SAMPLE_DOCUMENTS]
        self.nodes = SentenceSplitter().get_nodes_from_documents(self.documents)
        
        print("Initializing Indexes...")
        self.vector_index = None
        self.document_summary_index = None
        self.keyword_table_index = None
        
        print("Advanced Retrievers Lab Initialized!")
        print(f"Loaded {len(self.documents)} documents")
        print(f"Created {len(self.nodes)} nodes")
        
    def init_vectore_index(self):
        print("Initializing Vectore Index...")
        self.vector_index = VectorStoreIndex.from_documents(self.documents)
        print("Initializing Vectore Index is completed...")
        return self.vector_index
        
    def init_document_summary_index(self):
        print("Initializing Document Summary Index...")
        self.document_summary_index = DocumentSummaryIndex.from_documents(self.documents)
        print("Initializing Document Summary Index is completed...")
        return self.document_summary_index
        
    def init_keyword_table_index(self):
        print("Initializing Keyword Table Index...")
        self.keyword_table_index = KeywordTableIndex.from_documents(self.documents)
        print("Initializing Keyword Table Index is completed...")
        return self.keyword_table_index

In [56]:
lab = AdvanceRetrieverIndex()

Initializing Advanced Retrievers Lab...
Initializing Indexes...
Advanced Retrievers Lab Initialized!
Loaded 10 documents
Created 10 nodes


### Vectore Index Retriever

## 1. Vector Index Retriever - The Foundation

The Vector Index Retriever uses vector embeddings to find semantically related content, making it ideal for general-purpose search and widely used in retrieval-augmented generation (RAG) pipelines.

**How it works**: 
- Documents are split into nodes and embedded using the configured embedding model
- Query is converted to an embedding vector
- Returns nodes ranked by cosine similarity to the query embedding
- Generates embeddings in batches of 2048 nodes by default

**When to use:**
- General-purpose semantic search (most common use case)
- Finding conceptually related content based on meaning rather than exact keywords
- RAG pipelines where semantic understanding is crucial
- When exact keyword matching isn't the primary requirement

**Key characteristics from authoritative source:**
- **Stores embeddings for each document chunk** (VectorStoreIndex foundation)
- **Best for semantic retrieval** based on meaning and context
- **Commonly used in LLM pipelines** for retrieval-augmented generation

**Strengths**: 
- Excellent semantic understanding and context awareness
- Handles synonyms and related concepts effectively
- Works well with natural language queries

**Limitations**: 
- May miss exact keyword matches when specific terms are crucial
- Requires a good embedding model for optimal performance
- Can be computationally intensive for large document collections


In [ ]:
print("=" * 60)
print("1. VECTOR INDEX RETRIEVER")
print("=" * 60)

vector_store_index = lab.init_vectore_index()

vector_index_retriever = VectorIndexRetriever(
    index=vector_store_index,
    similarity_top_k=3
)

alt_retriever = vector_store_index.as_retriever(similarity_top_k=3)

query = DEMO_QUERIES['basic']
nodes = vector_index_retriever.retrieve(query)

print(f"Query: {query}")
print(f"Retrieved {len(nodes)} nodes:")

for i, node in enumerate(nodes,1):
    print(f"{i}. Score:{node.score:.4f}")
    print(f"     Text: {node.text[:100]}")
    print()

1. VECTOR INDEX RETRIEVER
Initializing Vectore Index...
Initializing Vectore Index is completed...
Query: What is machine learning?
Retrieved 3 nodes:
1. Score:0.7760
     Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr

2. Score:0.6485
     Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re

3. Score:0.5983
     Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in 



## 2. BM25 Retriever - Advanced Keyword-Based Search

BM25 is a keyword-based retrieval method that improves on TF-IDF by addressing some of its key limitations. It's widely used in production search systems including Elasticsearch and Apache Lucene.

### Understanding TF-IDF: The Foundation

Before diving into BM25, let's understand **TF-IDF** (Term Frequency-Inverse Document Frequency), which BM25 builds upon:

**Term Frequency (TF)**: Measures how often a word appears in a document
- Example: If "neural" appears 3 times in a 100-word document, TF = 3/100 = 0.03

**Inverse Document Frequency (IDF)**: Measures how rare a word is across all documents
- Example: If "neural" appears in only 2 out of 1000 documents, IDF = log(1000/2) = 6.21
- Common words like "the" have low IDF; rare technical terms have high IDF

**TF-IDF Score**: TF × IDF
- Highlights words that are frequent in one document but rare across the collection
- Developed by Karen Spärck Jones, who pioneered the concept of term specificity

### How BM25 Improves Upon TF-IDF

**Key BM25 Improvements:**

1. **Term Frequency Saturation**: BM25 reduces the impact of repeated terms using term frequency saturation
   - Problem: In TF-IDF, if a word appears 100 times vs 10 times, the score increases linearly
   - Solution: BM25 uses a saturation function that plateaus after a certain frequency

2. **Document Length Normalization**: BM25 adjusts for document length, making it more effective for keyword-based search
   - Problem: In TF-IDF, longer documents have unfair advantages
   - Solution: BM25 normalizes scores based on document length relative to average

3. **Tunable Parameters**: Allows fine-tuning for different types of content
   - k1 ≈ 1.2: Controls term frequency saturation (how quickly scores plateau)
   - b ≈ 0.75: Controls document length normalization (0=none, 1=full)

### When to Use BM25

**Ideal for:**
- Technical documentation where exact terms matter
- Legal documents with specific terminology
- Product catalogs with precise specifications
- Academic papers with specialized vocabulary
- Applications requiring keyword-based retrieval rather than semantic similarity

**Advantages:**
- Excellent precision for exact term matches
- Fast computational performance
- Proven effectiveness in production systems
- No training required (unlike neural approaches)
- Interpretable scoring mechanism

**Limitations:**
- No semantic understanding (doesn't handle synonyms)
- Struggles with typos and variations
- Limited context understanding
- Requires careful parameter tuning for optimal performance


In [52]:
import Stemmer
from llama_index.retrievers.bm25 import BM25Retriever

bm25_retriver = BM25Retriever.from_defaults(
    nodes=lab.nodes,
    similarity_top_k=3,
    stemmer=Stemmer.Stemmer('english'),
    language="english"
)

query = DEMO_QUERIES['technical']
nodes = bm25_retriver.retrieve(query)

print(f"Query: {query}")
print("BM25 analyzes exact keyword matches with sophisticated scoring")
print(f"Retrieved {len(nodes)} nodes:")

for node in nodes:
    print(node)

Query: neural networks deep learning
BM25 analyzes exact keyword matches with sophisticated scoring
Retrieved 3 nodes:
Node ID: a8d57917-4017-4881-984f-cd83621c35e7
Text: Deep learning uses neural networks with multiple layers to model
and understand complex patterns in data.
Score:  2.520

Node ID: f733b7dc-f141-4939-90ee-77d482286f38
Text: Reinforcement learning is a type of machine learning where
agents learn to make decisions through rewards and penalties.
Score:  0.337

Node ID: cf2de21b-a6d3-4923-82a1-40f274caa1a6
Text: Machine learning is a subset of artificial intelligence that
focuses on algorithms that can learn from data.
Score:  0.302



## 3. Document Summary Index Retrievers

Document Summary Index Retrievers use document summaries instead of the actual documents to find relevant content, making them efficient for large collections. **They return the original documents, not their summaries.**

**How it works (from authoritative source)**:
- **Generates and stores summaries of documents** at indexing time
- **Uses summaries to filter documents** before retrieving full content
- **Two-stage Process**: First uses summaries to filter documents, then returns full document content
- **Especially useful for large, diverse corpora** that cannot fit in the context window of an LLM

**Two Retrieval Options**: 
1. **DocumentSummaryIndexLLMRetriever**: 
   - Uses a large language model to analyze the query against document summaries
   - Provides intelligent document selection but can be more time-consuming and expensive
   - Best for complex queries requiring nuanced understanding

2. **DocumentSummaryIndexEmbeddingRetriever**: 
   - Uses semantic similarity between the query and summary embeddings
   - Faster and more cost-effective than LLM-based approach
   - Good for straightforward similarity matching

**When to use (based on authoritative guidance):**
- Large document collections where documents cover different topics
- When you need efficient document-level filtering before detailed retrieval
- Multi-document QA where documents have distinct subject matters
- Large and diverse document sets that cannot fit in the context window of an LLM

**Configuration Parameters:**
- `choice_top_k` (LLM retriever): Number of documents to select
- `similarity_top_k` (Embedding retriever): Number of documents to select
- Default is 1, increase for multiple document retrieval

**Key Point**: **Returns original documents, not their summaries** - the summaries are only used for filtering

**Strengths**: 
- Efficient document selection and reduces search space
- Good for heterogeneous collections with diverse topics
- Returns original documents with full context intact

**Limitations**: 
- Requires LLM for summary generation during indexing
- May lose some detail present in original documents during summary creation
- LLM-based version can be slower and more expensive than other options


In [53]:
from llama_index.core.indices.document_summary import DocumentSummaryIndexLLMRetriever,DocumentSummaryIndexEmbeddingRetriever

In [57]:
print("=" * 60)
print("3. DOCUMENT SUMMARY INDEX RETRIEVERS")
print("=" * 60)

document_summary_index = lab.init_document_summary_index()

document_summary_retriever_llm = DocumentSummaryIndexLLMRetriever(
    index=document_summary_index,
    choice_top_k=3
)

document_summary_retriever_embedding = DocumentSummaryIndexEmbeddingRetriever(
    index=document_summary_index,
    similarity_top_k=3
)

query = DEMO_QUERIES["learning_types"]  # "different types of learning"


print(f"Query: {query}")

print("\nA) LLM-based Document Summary Retriever:")
print("Uses LLM to select relevant documents based on summaries")

try:
    
    nodes_llm = document_summary_retriever_llm.retrieve(query)
    print(f"Retrieved {len(nodes_llm)} nodes")
    
    for node in nodes_llm:
        print(node)

except Exception as e:
    print(f"LLM-based retrieval demo: {str(e)[:100]}...")
    
print("B) Embedding-based Document Summary Retriever:")
print("Uses vector similarity between query and document summaries")

try:
    nodes_embedding = document_summary_retriever_embedding.retrieve(query)
    print(f"Retrieved {len(nodes_llm)} nodes")
    
    for node in nodes_embedding:
        print(node)
except Exception as e:
    print(f"Embedding-based retrieval demo: {str(e)[:100]}...")
    
print("Document Summary Index workflow:")
print("1. Generates summaries for each document using LLM")
print("2. Uses summaries to select relevant documents")
print("3. Returns full content from selected documents")

Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


3. DOCUMENT SUMMARY INDEX RETRIEVERS
Initializing Document Summary Index...
current doc id: c066edeb-6354-474c-bea7-1e4709bf275b


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


current doc id: 748f204e-9084-4341-979f-f9ac08e67e1e


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


...

current doc id: 622f80a9-80e0-44c0-a6c6-ea613d9a2ac3


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


current doc id: 8da6346f-9aba-4d5c-8707-ba4ed385632f


Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Initializing Document Summary Index is completed...
Query: different types of learning

A) LLM-based Document Summary Retriever:
Uses LLM to select relevant documents based on summaries
Retrieved 3 nodes
Node ID: 007a981a-7343-4cc7-b7c7-0eae3703cb6d
Text: Machine learning is a subset of artificial intelligence that
focuses on algorithms that can learn from data.
Score:  9.000

Node ID: 87c5938c-9629-44ce-b638-0bd7aae60404
Text: Reinforcement learning is a type of machine learning where
agents learn to make decisions through rewards and penalties.
Score:  8.000

Node ID: 6dc4419f-ff28-4363-bfbf-c69be7cbc9e7
Text: Supervised learning uses labeled training data to learn a
mapping from inputs to outputs.
Score:  7.000

B) Embedding-based Document Summary Retriever:
Uses vector similarity between query and document summaries
Retrieved 3 nodes
Node ID: 87c5938c-9629-44ce-b638-0bd7aae60404
Text: Reinforcement learning is a type of machine learning where
agents learn to make decisions through 

## 4. Auto Merging Retriever - Hierarchical Context Preservation

Auto Merging Retriever is designed to preserve context in long documents using a hierarchical structure. **It uses hierarchical chunking to break documents into parent and child nodes, and if enough child nodes from the same parent are retrieved, the retriever returns the parent node instead.**

**How it works (from authoritative source)**:
- **Uses hierarchical chunking** to break documents into parent and child nodes
- **Retrieves parent if enough children match** - intelligent merging logic
- **Preserves context in long documents** by consolidating related content
- **Dual Storage**: Smaller child chunks are indexed in the vector store for precise matching, while larger parent chunks are stored in the docstore

**Key behavior pattern**:
- Child chunks enable precise matching for specific queries
- When multiple child chunks from the same parent are retrieved, the system returns the parent chunk
- This **helps consolidate related content and preserve broader context**

**When to use (based on authoritative guidance):**
- Long documents where small chunks lose important surrounding context
- Legal documents, research papers, technical specifications that need context preservation
- When you need both precise matching and comprehensive context
- Documents with natural hierarchical structure (sections, subsections)

**Configuration:**
- `chunk_sizes`: List of chunk sizes from largest to smallest (e.g., [512, 256, 128])
- `chunk_overlap`: Overlap between chunks to maintain continuity
- Storage context manages both vector store (child nodes) and docstore (parent nodes)

**Strengths**: 
- Automatically preserves context without manual intervention
- Reduces information fragmentation in long documents
- Intelligent merging based on retrieval patterns
- Maintains granular search capability while providing broader context

**Limitations**: 
- More complex setup compared to basic retrievers
- Requires hierarchical document structure to be effective
- Higher storage overhead due to multiple chunk levels
- May not be suitable for very short documents

*Based on: https://docs.llamaindex.ai/en/stable/examples/retrievers/auto_merging_retriever/*


In [60]:
from llama_index.core.node_parser import HierarchicalNodeParser
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core import StorageContext
from llama_index.core.retrievers import AutoMergingRetriever

In [61]:
print("=" * 60)
print("4. AUTO MERGING RETRIEVER")
print("=" * 60)

node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[512,256,128]
)

hier_nodes = node_parser.get_nodes_from_documents(lab.documents)

docstore = SimpleDocumentStore()
docstore.add_documents(hier_nodes)

storage_context = StorageContext.from_defaults(docstore=docstore)

#Create Base Index
base_index = VectorStoreIndex(
    nodes=hier_nodes, storage_context=storage_context
)
base_retriever = base_index.as_retriever(similarity_top_k = 3)

auto_merge_retriever = AutoMergingRetriever(
    vector_retriever=base_retriever,
    storage_context=storage_context,
    verbose=True
)

query = DEMO_QUERIES['advanced']
nodes = auto_merge_retriever.retrieve(query)

print(f"Query: {query}")
print(f"Auto-merged to {len(nodes)} nodes")
for i, node in enumerate(nodes[:3], 1):
    print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Auto-merged)")
    print(f"   Text: {node.text[:120]}...")
    print()






4. AUTO MERGING RETRIEVER
> Merging 1 nodes into parent node.
> Parent node id: a9df3c27-6653-4358-827b-e0b878cad3cc.
> Parent node text: Deep learning uses neural networks with multiple layers to model and understand complex patterns ...

Query: How do neural networks work in deep learning?
Auto-merged to 1 nodes
1. Score: 0.7332
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in data....

